# Retail Customer Segmentation — Analysis
**Датасет:** retail_customer_segmentation.csv  
**Строк:** 50 000 | **Колонок:** 14  
**Стек:** Python · Pandas · Seaborn · Scikit-learn · SciPy

## Часть 1 — Загрузка и первичный осмотр

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
# Загружаем датасет
df = pd.read_csv('retail_customer_segmentation.csv')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# Общая статистика по числовым колонкам
df.describe()

In [ ]:
# Уникальные значения категориальных колонок
print('Сегменты:', df['customer_segment'].unique())
print('Регионы:', df['region'].unique())
print('Способы оплаты:', df['payment_method'].unique())

## Часть 2 — Очистка данных

In [ ]:
# Проверяем пропуски
df.isnull().sum()

In [ ]:
# Фиксируем размер до очистки
before = len(df)
print(f'Строк до очистки: {before}')

In [ ]:
# Удаляем дубликаты
df = df.drop_duplicates()
print(f'Удалено дублей: {before - len(df)}')

In [ ]:
# Заполняем пропуски в числовых колонках медианой
num_cols = ['annual_income', 'avg_monthly_spend', 'purchase_frequency',
            'discount_usage_rate', 'return_rate',
            'browsing_time_minutes', 'support_interactions']

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

print('Пропуски после заполнения:')
df.isnull().sum()

In [ ]:
# Создаём новый признак — расчётный LTV клиента
df['estimated_ltv'] = df['avg_order_value'] * df['purchase_frequency'] * df['months_active']

df[['customer_id', 'customer_segment', 'estimated_ltv']].head()

In [ ]:
print(f'Строк после очистки: {len(df)}')
print(f'Удалено всего: {before - len(df)}')

## Часть 3 — Разведочный анализ (EDA)

In [ ]:
# Распределение клиентов по сегментам
segment_counts = df['customer_segment'].value_counts()

plt.figure(figsize=(8, 5))
sns.barplot(x=segment_counts.index, y=segment_counts.values, palette='viridis')
plt.title('Количество клиентов по сегментам')
plt.xlabel('Сегмент')
plt.ylabel('Количество клиентов')
plt.tight_layout()
plt.show()

print(segment_counts)
# Вывод: Occasional — самый большой сегмент (44%), High_Value — наименьший (10.5%)

In [ ]:
# Распределение среднего ежемесячного расхода
plt.figure(figsize=(8, 5))
sns.histplot(df['avg_monthly_spend'], bins=50, kde=True, color='steelblue')
plt.title('Распределение среднего ежемесячного расхода')
plt.xlabel('Avg Monthly Spend')
plt.ylabel('Количество клиентов')
plt.tight_layout()
plt.show()

# Вывод: большинство клиентов тратят 150–500 в месяц, есть выбросы выше 2000

In [ ]:
# Средний чек по сегментам
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='customer_segment', y='avg_monthly_spend',
            order=['Occasional','Regular','Loyal','High_Value'], palette='Set2')
plt.title('Средний ежемесячный расход по сегментам')
plt.xlabel('Сегмент')
plt.ylabel('Avg Monthly Spend')
plt.tight_layout()
plt.show()

# Вывод: High_Value тратит значительно больше остальных сегментов

In [ ]:
# Средний расход по регионам
plt.figure(figsize=(7, 5))
sns.barplot(data=df, x='region', y='avg_monthly_spend',
            order=['Urban','Semi-Urban','Rural'], palette='Blues_d')
plt.title('Средний ежемесячный расход по регионам')
plt.xlabel('Регион')
plt.ylabel('Avg Monthly Spend')
plt.tight_layout()
plt.show()

# Вывод: Urban-клиенты тратят больше всего, Rural — меньше всего

In [ ]:
# Корреляционная матрица числовых признаков
num_df = df[['age', 'annual_income', 'months_active', 'avg_monthly_spend',
             'purchase_frequency', 'avg_order_value', 'discount_usage_rate',
             'return_rate', 'browsing_time_minutes', 'estimated_ltv']]

plt.figure(figsize=(10, 7))
sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Корреляционная матрица')
plt.tight_layout()
plt.show()

# Вывод: avg_order_value и purchase_frequency сильнее всего влияют на estimated_ltv

## Часть 4 — Ключевые метрики

In [ ]:
# Средний LTV по сегментам
ltv_by_segment = df.groupby('customer_segment')['estimated_ltv'].mean().sort_values(ascending=False).reset_index()
ltv_by_segment.columns = ['Сегмент', 'Средний LTV']
ltv_by_segment['Средний LTV'] = ltv_by_segment['Средний LTV'].round(0)

print(ltv_by_segment)
# Вывод: High_Value имеет LTV в разы выше Occasional

In [ ]:
# Retention Rate — доля клиентов активных более 12 месяцев
retained = df[df['months_active'] > 12]['customer_id'].nunique()
total = df['customer_id'].nunique()
retention_rate = retained / total

print(f'Retention Rate: {retention_rate:.1%}')
# Вывод: показывает какая доля клиентов остаётся дольше года

In [ ]:
# Сводная таблица метрик по сегментам
summary = df.groupby('customer_segment').agg(
    customers=('customer_id', 'count'),
    avg_monthly_spend=('avg_monthly_spend', 'mean'),
    avg_order_value=('avg_order_value', 'mean'),
    avg_return_rate=('return_rate', 'mean'),
    avg_discount_usage=('discount_usage_rate', 'mean'),
    avg_ltv=('estimated_ltv', 'mean')
).round(2)

summary

## Часть 5 — Линейная регрессия

In [ ]:
# Цель: предсказать avg_monthly_spend по характеристикам клиента
features = ['age', 'annual_income', 'months_active', 'purchase_frequency', 'avg_order_value']
target = 'avg_monthly_spend'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
r2  = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f'R²  = {r2:.3f}   — модель объясняет {r2*100:.1f}% вариации расходов')
print(f'MAE = {mae:.2f}  — средняя ошибка предсказания')

In [ ]:
# Важность признаков
coef_df = pd.DataFrame({'Признак': features, 'Коэффициент': model.coef_})
coef_df = coef_df.reindex(coef_df['Коэффициент'].abs().sort_values(ascending=False).index)

plt.figure(figsize=(8, 5))
sns.barplot(data=coef_df, x='Коэффициент', y='Признак', palette='RdBu')
plt.title('Важность признаков в линейной регрессии')
plt.tight_layout()
plt.show()

# Вывод: purchase_frequency и avg_order_value — главные предикторы расходов

## Часть 6 — Проверка гипотез

In [ ]:
# Гипотеза 1: Urban-клиенты тратят больше, чем Rural
urban = df[df['region'] == 'Urban']['avg_monthly_spend']
rural = df[df['region'] == 'Rural']['avg_monthly_spend']

t_stat, p_value = stats.ttest_ind(urban, rural)

print(f'Среднее Urban: {urban.mean():.2f}')
print(f'Среднее Rural: {rural.mean():.2f}')
print(f'p-value = {p_value:.4f}')

if p_value < 0.05:
    print('✅ Гипотеза подтверждена — разница статистически значима')
else:
    print('❌ Гипотеза не подтверждена — разница случайна')

In [ ]:
# Гипотеза 2: Клиенты с высоким discount_usage тратят меньше
high_discount = df[df['discount_usage_rate'] > 0.5]['avg_order_value']
low_discount  = df[df['discount_usage_rate'] <= 0.5]['avg_order_value']

t_stat, p_value = stats.ttest_ind(high_discount, low_discount)

print(f'Средний чек (высокий дисконт): {high_discount.mean():.2f}')
print(f'Средний чек (низкий дисконт):  {low_discount.mean():.2f}')
print(f'p-value = {p_value:.4f}')

if p_value < 0.05:
    print('✅ Гипотеза подтверждена — скидки снижают средний чек')
else:
    print('❌ Гипотеза не подтверждена — разница случайна')

## Часть 7 — Выводы и рекомендации

In [ ]:
print("""
КЛЮЧЕВЫЕ ВЫВОДЫ
===============
1. ...
2. ...
3. ...

РЕКОМЕНДАЦИИ БИЗНЕСУ
====================
1. ...
2. ...
3. ...
""")
# Заполни после получения результатов